In [ ]:
# taking le's numerous feature combinations and running many metrics and tests on them to find the 'best' combinations

In [1]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')

# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import ast
import re

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

# importing util
import os
os.chdir('/content/drive/MyDrive/DSSI/Project/code')
import utils

print('finished imports!')

Mounted at /content/drive
finished imports!


In [2]:
env_model_results = pd.read_csv('/content/drive/MyDrive/DSSI/Project/data/env_only_all_model_results.csv', parse_dates=['Start_Date','End_Date'])
env_model_results

,County,Env_Combination,Number_Env_Features,Exog_Lag,Number_Model_Features,Number_Observations,Start_Date,End_Date,RMSE_Mean,RMSE_Std,MAE_Mean,MAE_Std,R2_Mean,R2_Std,Fold_RMSEs,Fold_MAEs,Fold_R2s
0,Fresno,"('Avg Rel Hum (%)', 'Wind Run (miles)', 'winde...",4,3,4,194,2006-11-01,2022-12-01,0.237507,0.057798,0.189117,0.051457,0.021969,0.179823,"[np.float64(0.2842224928349675), np.float64(0....","[0.22389265697740393, 0.17917954982044798, 0.2...","[0.004536894596836039, -0.31881660795101907, 0..."
1,Fresno,"('Avg Rel Hum (%)', 'Wind Run (miles)', 'winde...",4,3,4,194,2006-11-01,2022-12-01,0.237647,0.061666,0.191240,0.055447,0.026118,0.194619,"[np.float64(0.2858460996223104), np.float64(0....","[0.22436487458064897, 0.1819695734781298, 0.27...","[-0.006868655930156953, -0.33410657507375885, ..."
2,Fresno,"('Avg Rel Hum (%)', 'Wind Run (miles)', 'winde...",3,3,3,194,2006-11-01,2022-12-01,0.237784,0.061174,0.192071,0.054564,0.026655,0.172096,"[np.float64(0.285757330289345), np.float64(0.2...","[0.22441988559918832, 0.17831793369631543, 0.2...","[-0.006243388131978556, -0.29129159320446707, ..."
3,Fresno,"('Avg Soil Temp (F)', 'AQI_PM25', 'AQI_PM10', ...",4,3,4,194,2006-11-01,2022-12-01,0.238024,0.057785,0.192394,0.052139,0.021575,0.130328,"[np.float64(0.280753649798333), np.float64(0.2...","[0.2264374445113102, 0.17293286092152121, 0.27...","[0.028687222842491478, -0.232383872280993, 0.0..."
4,Fresno,"('Avg Soil Temp (F)', 'AQI_PM10', 'fungicide_l...",3,3,3,194,2006-11-01,2022-12-01,0.238041,0.056137,0.191550,0.050437,0.017550,0.132735,"[np.float64(0.27950820254313347), np.float64(0...","[0.22227419699037376, 0.17362846398070697, 0.2...","[0.03728576198812927, -0.24275542051139465, 0...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26923,Ventura,"('Wind Run (miles)', 'Avg Soil Temp (F)', 'FIR...",4,5,4,192,2007-01-01,2022-12-01,0.307223,0.040827,0.236786,0.031880,-0.245582,0.236904,"[np.float64(0.2802062619642818), np.float64(0....","[0.21071937026088397, 0.2544950764644225, 0.28...","[-0.11522913678531288, -0.4232362734191557, -0..."
26924,Ventura,"('Wind Run (miles)', 'Avg Soil Temp (F)', 'rod...",3,5,3,192,2007-01-01,2022-12-01,0.307223,0.040827,0.236786,0.031880,-0.245582,0.236904,"[np.float64(0.2802062619642806), np.float64(0....","[0.2107193702608826, 0.2544950764644258, 0.283...","[-0.11522913678530378, -0.4232362734191908, -0..."
26925,Ventura,"('Wind Run (miles)', 'Avg Soil Temp (F)', 'fun...",4,5,4,192,2007-01-01,2022-12-01,0.307358,0.040234,0.236134,0.032228,-0.246111,0.230368,"[np.float64(0.28098293789804285), np.float64(0...","[0.20763435318643936, 0.2548635634408851, 0.28...","[-0.12142009092522921, -0.42585683566078725, -..."
26926,Ventura,"('Wind Run (miles)', 'Avg Soil Temp (F)', 'AQI...",4,5,4,192,2007-01-01,2022-12-01,0.308075,0.041424,0.237551,0.032904,-0.252542,0.241613,"[np.float64(0.28281801906121895), np.float64(0...","[0.21560394449058096, 0.2563307174184064, 0.28...","[-0.13611576529742297, -0.43852034053972133, -..."


# convert Env_Combination to tuple

In [3]:
def parse_feature_combination(value):
    """
    Convert a CSV string such as:
    "('Precipitation', 'AQI_PM10')"

    into:
    ('Precipitation', 'AQI_PM10')
    """
    if isinstance(value, tuple):
        return value

    if isinstance(value, list):
        return tuple(value)

    if pd.isna(value):
        return tuple()

    try:
        parsed = ast.literal_eval(value)

        if isinstance(parsed, str):
            return (parsed,)

        return tuple(parsed)

    except (ValueError, SyntaxError, TypeError):
        # Fallback for strings separated with +
        return tuple(
            feature.strip()
            for feature in str(value).split("+")
            if feature.strip()
        )


env_model_results["Env_Combination"] = (
    env_model_results["Env_Combination"]
    .apply(parse_feature_combination)
)

# choose top n (20) models to compare

In [4]:
TOP_N = 20

competitive_models = (
    env_model_results
    .sort_values(
        ["County", "RMSE_Mean", "RMSE_Std", "MAE_Mean"],
        ascending=[True, True, True, True]
    )
    .groupby("County", group_keys=False)
    .head(TOP_N)
    .copy()
)

In [5]:
competitive_models["Model_Rank"] = (
    competitive_models
    .groupby("County")
    .cumcount() + 1
)

# expand models into one row per feature (explode)

In [6]:
feature_model_rows = (
    competitive_models
    .explode("Env_Combination")
    .rename(columns={"Env_Combination": "Feature"})
    .dropna(subset=["Feature"])
    .copy()
)

# raw feature-freq counts

In [7]:
feature_frequency = (
    feature_model_rows
    .groupby(["County", "Feature"])
    .size()
    .rename("Raw_Frequency")
    .reset_index()
)

In [8]:
number_competitive_models = (
    competitive_models
    .groupby("County")
    .size()
    .rename("Number_Competitive_Models")
    .reset_index()
)

feature_frequency = feature_frequency.merge(
    number_competitive_models,
    on="County",
    how="left"
)

feature_frequency["Frequency_Percentage"] = (
    100
    * feature_frequency["Raw_Frequency"]
    / feature_frequency["Number_Competitive_Models"]
)

In [9]:
feature_frequency = (
    feature_frequency
    .sort_values(
        ["County", "Raw_Frequency"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

display(feature_frequency)

,County,Feature,Raw_Frequency,Number_Competitive_Models,Frequency_Percentage
0,Fresno,windeventcount,13,20,65.0
1,Fresno,AQI_PM10,11,20,55.0
2,Fresno,Wind Run (miles),11,20,55.0
3,Fresno,fungicide_lbs_prd_used,10,20,50.0
4,Fresno,Avg Soil Temp (F),7,20,35.0
...,...,...,...,...,...
65,Ventura,windeventcount,5,20,25.0
66,Ventura,AQI_PM25,2,20,10.0
67,Ventura,Avg Soil Temp (F),2,20,10.0
68,Ventura,Avg Wind Speed (mph),2,20,10.0


# rank-weighted frequency

gives higher ranks more value

In [10]:
competitive_models["Rank_Weight"] = (
    TOP_N - competitive_models["Model_Rank"] + 1
)

In [11]:
feature_model_rows = (
    competitive_models
    .explode("Env_Combination")
    .rename(columns={"Env_Combination": "Feature"})
    .dropna(subset=["Feature"])
    .copy()
)

In [12]:
weighted_frequency = (
    feature_model_rows
    .groupby(["County", "Feature"])
    .agg(
        Rank_Weighted_Frequency=("Rank_Weight", "sum"),
        Average_Model_Rank=("Model_Rank", "mean"),
        Best_Model_Rank=("Model_Rank", "min")
    )
    .reset_index()
)

In [13]:
maximum_weight = sum(range(1, TOP_N + 1))

weighted_frequency["Weighted_Frequency_Percentage"] = (
    100
    * weighted_frequency["Rank_Weighted_Frequency"]
    / maximum_weight
)

In [14]:
feature_screening = feature_frequency.merge(
    weighted_frequency,
    on=["County", "Feature"],
    how="left"
)

# measure performance of models containing each feature

In [15]:
feature_performance = (
    feature_model_rows
    .groupby(["County", "Feature"])
    .agg(
        Mean_RMSE=("RMSE_Mean", "mean"),
        Median_RMSE=("RMSE_Mean", "median"),
        Best_RMSE=("RMSE_Mean", "min"),
        Worst_RMSE=("RMSE_Mean", "max"),

        Mean_MAE=("MAE_Mean", "mean"),
        Mean_R2=("R2_Mean", "mean"),

        Mean_RMSE_Std=("RMSE_Std", "mean"),
        Median_RMSE_Std=("RMSE_Std", "median"),

        Mean_R2_Std=("R2_Std", "mean")
    )
    .reset_index()
)

In [16]:
feature_screening = feature_screening.merge(
    feature_performance,
    on=["County", "Feature"],
    how="left"
)

# compare models w versus w/o each feature

In [17]:
comparison_rows = []

for county, county_results in env_model_results.groupby("County"):

    all_features = sorted({
        feature
        for combination in county_results["Env_Combination"]
        for feature in combination
    })

    for feature in all_features:

        stratum_improvements = []

        strata = county_results.groupby(
            ["Number_Env_Features", "Exog_Lag"]
        )

        for (model_size, lag), stratum in strata:

            contains_feature = stratum[
                stratum["Env_Combination"].apply(
                    lambda combination: feature in combination
                )
            ]

            excludes_feature = stratum[
                stratum["Env_Combination"].apply(
                    lambda combination: feature not in combination
                )
            ]

            if contains_feature.empty or excludes_feature.empty:
                continue

            improvement = (
                excludes_feature["RMSE_Mean"].mean()
                - contains_feature["RMSE_Mean"].mean()
            )

            stratum_improvements.append(improvement)

        if stratum_improvements:
            mean_improvement = np.mean(stratum_improvements)
            median_improvement = np.median(stratum_improvements)
            positive_rate = np.mean(
                np.array(stratum_improvements) > 0
            )
            number_comparisons = len(stratum_improvements)
        else:
            mean_improvement = np.nan
            median_improvement = np.nan
            positive_rate = np.nan
            number_comparisons = 0

        comparison_rows.append({
            "County": county,
            "Feature": feature,
            "Matched_RMSE_Improvement": mean_improvement,
            "Median_Matched_Improvement": median_improvement,
            "Positive_Improvement_Rate": positive_rate,
            "Number_Matched_Comparisons": number_comparisons
        })


feature_comparison = pd.DataFrame(comparison_rows)

In [18]:
feature_screening = feature_screening.merge(
    feature_comparison,
    on=["County", "Feature"],
    how="left"
)

# check fold stability

In [19]:
feature_screening["RMSE_Coefficient_of_Variation"] = (
    feature_screening["Mean_RMSE_Std"]
    / feature_screening["Mean_RMSE"]
)

In [20]:
feature_screening["Stable_Performance"] = (
    feature_screening["RMSE_Coefficient_of_Variation"]
    <= feature_screening
        .groupby("County")["RMSE_Coefficient_of_Variation"]
        .transform("median")
)

# check consistency across lags

In [21]:
lag_consistency = (
    feature_model_rows
    .groupby(["County", "Feature"])
    .agg(
        Number_Distinct_Lags=("Exog_Lag", "nunique"),
        Minimum_Competitive_Lag=("Exog_Lag", "min"),
        Maximum_Competitive_Lag=("Exog_Lag", "max"),
        Most_Common_Lag=(
            "Exog_Lag",
            lambda values: values.mode().iloc[0]
        )
    )
    .reset_index()
)

In [22]:
lag_lists = (
    feature_model_rows
    .groupby(["County", "Feature"])["Exog_Lag"]
    .apply(lambda values: sorted(values.unique().tolist()))
    .rename("Competitive_Lags")
    .reset_index()
)

lag_consistency = lag_consistency.merge(
    lag_lists,
    on=["County", "Feature"],
    how="left"
)

In [23]:
feature_screening = feature_screening.merge(
    lag_consistency,
    on=["County", "Feature"],
    how="left"
)

# add county-specific ranks

In [24]:
feature_screening["Frequency_Rank"] = (
    feature_screening
    .groupby("County")["Raw_Frequency"]
    .rank(ascending=False, method="min")
)

feature_screening["Weighted_Frequency_Rank"] = (
    feature_screening
    .groupby("County")["Rank_Weighted_Frequency"]
    .rank(ascending=False, method="min")
)

feature_screening["Performance_Rank"] = (
    feature_screening
    .groupby("County")["Mean_RMSE"]
    .rank(ascending=True, method="min")
)

feature_screening["Stability_Rank"] = (
    feature_screening
    .groupby("County")["RMSE_Coefficient_of_Variation"]
    .rank(ascending=True, method="min")
)

feature_screening["Improvement_Rank"] = (
    feature_screening
    .groupby("County")["Matched_RMSE_Improvement"]
    .rank(ascending=False, method="min")
)

feature_screening["Lag_Consistency_Rank"] = (
    feature_screening
    .groupby("County")["Number_Distinct_Lags"]
    .rank(ascending=False, method="min")
)

# create an overall screening score

In [25]:
feature_screening["Screening_Score"] = (
    0.30 * feature_screening["Frequency_Rank"]
    + 0.25 * feature_screening["Weighted_Frequency_Rank"]
    + 0.20 * feature_screening["Performance_Rank"]
    + 0.15 * feature_screening["Improvement_Rank"]
    + 0.075 * feature_screening["Stability_Rank"]
    + 0.025 * feature_screening["Lag_Consistency_Rank"]
)

In [26]:
feature_screening = (
    feature_screening
    .sort_values(
        [
            "County",
            "Screening_Score",
            "Mean_RMSE"
        ],
        ascending=[True, True, True]
    )
    .reset_index(drop=True)
)

feature_screening["Overall_Feature_Rank"] = (
    feature_screening
    .groupby("County")
    .cumcount() + 1
)

# view results

In [27]:
screening_columns = [
    "County",
    "Overall_Feature_Rank",
    "Feature",
    "Raw_Frequency",
    "Frequency_Percentage",
    "Rank_Weighted_Frequency",
    "Weighted_Frequency_Percentage",
    "Average_Model_Rank",
    "Mean_RMSE",
    "Best_RMSE",
    "Mean_RMSE_Std",
    "RMSE_Coefficient_of_Variation",
    "Matched_RMSE_Improvement",
    "Positive_Improvement_Rate",
    "Number_Distinct_Lags",
    "Competitive_Lags",
    "Most_Common_Lag",
    "Screening_Score"
]

display(
    feature_screening[screening_columns]
)

,County,Overall_Feature_Rank,Feature,Raw_Frequency,Frequency_Percentage,Rank_Weighted_Frequency,Weighted_Frequency_Percentage,Average_Model_Rank,Mean_RMSE,Best_RMSE,Mean_RMSE_Std,RMSE_Coefficient_of_Variation,Matched_RMSE_Improvement,Positive_Improvement_Rate,Number_Distinct_Lags,Competitive_Lags,Most_Common_Lag,Screening_Score
0,Fresno,1,windeventcount,13,65.0,145,69.047619,9.846154,0.238418,0.237507,0.056652,0.237616,0.006533,0.875000,1,[3],3,2.475
1,Fresno,2,fungicide_lbs_prd_used,10,50.0,128,60.952381,8.200000,0.238320,0.237507,0.055434,0.232605,0.006227,0.750000,1,[3],3,3.400
2,Fresno,3,AQI_PM10,11,55.0,105,50.000000,11.454545,0.238605,0.238024,0.055673,0.233327,0.007531,0.875000,2,"[1, 3]",3,3.500
3,Fresno,4,Wind Run (miles),11,55.0,118,56.190476,10.272727,0.238446,0.237507,0.057902,0.242829,0.006477,1.000000,2,"[1, 3]",3,3.650
4,Fresno,5,Avg Soil Temp (F),7,35.0,85,40.476190,8.857143,0.238399,0.238024,0.056035,0.235047,0.007867,0.958333,1,[3],3,3.950
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,Ventura,6,rodent_lbs_prd_used,6,30.0,45,21.428571,13.500000,0.272142,0.271890,0.023906,0.087845,-0.003041,0.291667,1,[3],3,6.075
66,Ventura,7,Precipitation,5,25.0,36,17.142857,13.800000,0.272105,0.271890,0.024080,0.088496,0.000571,0.625000,1,[3],3,6.150
67,Ventura,8,Avg Wind Speed (mph),2,10.0,27,12.857143,7.500000,0.271758,0.271758,0.015571,0.057297,-0.002881,0.166667,1,[5],5,6.625
68,Ventura,9,windeventcount,5,25.0,24,11.428571,16.200000,0.272324,0.271890,0.023640,0.086807,-0.000620,0.625000,1,[3],3,7.450


# extract top five features per county

In [28]:
top_5_features = (
    feature_screening[
        feature_screening["Overall_Feature_Rank"] <= 5
    ][screening_columns]
    .reset_index(drop=True)
)

display(top_5_features)

,County,Overall_Feature_Rank,Feature,Raw_Frequency,Frequency_Percentage,Rank_Weighted_Frequency,Weighted_Frequency_Percentage,Average_Model_Rank,Mean_RMSE,Best_RMSE,Mean_RMSE_Std,RMSE_Coefficient_of_Variation,Matched_RMSE_Improvement,Positive_Improvement_Rate,Number_Distinct_Lags,Competitive_Lags,Most_Common_Lag,Screening_Score
0,Fresno,1,windeventcount,13,65.0,145,69.047619,9.846154,0.238418,0.237507,0.056652,0.237616,0.006533,0.875000,1,[3],3,2.475
1,Fresno,2,fungicide_lbs_prd_used,10,50.0,128,60.952381,8.200000,0.238320,0.237507,0.055434,0.232605,0.006227,0.750000,1,[3],3,3.400
2,Fresno,3,AQI_PM10,11,55.0,105,50.000000,11.454545,0.238605,0.238024,0.055673,0.233327,0.007531,0.875000,2,"[1, 3]",3,3.500
3,Fresno,4,Wind Run (miles),11,55.0,118,56.190476,10.272727,0.238446,0.237507,0.057902,0.242829,0.006477,1.000000,2,"[1, 3]",3,3.650
4,Fresno,5,Avg Soil Temp (F),7,35.0,85,40.476190,8.857143,0.238399,0.238024,0.056035,0.235047,0.007867,0.958333,1,[3],3,3.950
5,Kern,1,Avg Rel Hum (%),15,75.0,160,76.190476,10.333333,0.224595,0.222282,0.042145,0.187648,0.006854,1.000000,3,"[4, 5, 6]",4,1.875
6,Kern,2,fungicide_lbs_prd_used,13,65.0,166,79.047619,8.230769,0.224325,0.222282,0.044597,0.198806,0.005349,0.833333,2,"[4, 5]",4,2.100
7,Kern,3,Avg Soil Temp (F),10,50.0,106,50.476190,10.400000,0.224847,0.223660,0.043026,0.191358,0.006973,1.000000,3,"[4, 5, 6]",6,3.050
8,Kern,4,rodent_lbs_prd_used,3,15.0,38,18.095238,8.333333,0.224522,0.223647,0.043063,0.191801,0.004260,0.791667,2,"[4, 5]",5,3.875
9,Kern,5,AQI_PM10,3,15.0,30,14.285714,11.000000,0.225095,0.224716,0.045122,0.200460,0.001806,0.666667,2,"[4, 6]",6,5.525


top 20 candidate combinations per county
* Rank_Weighted_Frequency = weighted version of frequency that gives more value based off higher-ranked models
    * Rank 1 gets 20 points, rank 2 gets 19...
* Average_Rodel_Rank = Average rank of models containing the feature (lower better)
* Matched_RMSE_Improvement = avg RMSE improvement for models containing the feature compared with models excluding it, matching on county, # features, lag
* Positive_Improvement_Rate = proportion of matched comparisons in which including the feature improved RMSE
* Number_Distinct_Lags = how many different lags of the feature appear among competitive models
* Screening_Score = combined weighted ranking score based on raw frequency, rank-weighted frequency, avg RMSE, matched RMSE improvement, fold stability, lag consistency (LOWER BETTER)

In [29]:
# saving
# (feature_screening[screening_columns]).to_csv('top10_features_per_county.csv', index=False)
# top_5_features.to_csv('top5_features_per_county.csv', index=False)

# using narrowed search fields, finding the best lags per feature

In [60]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score

In [61]:
selected_features_by_county = {
    "Fresno": [
        "windeventcount",
        "fungicide_lbs_prd_used",
        "AQI_PM10",
        "Avg Soil Temp (F)",
        "Wind Run (miles)"
    ],

    "Kern": [
        "Avg Rel Hum (%)",
        "fungicide_lbs_prd_used",
        "Avg Soil Temp (F)"
    ],

    "Los Angeles": [
        "Avg Wind Speed (mph)",
        "Precipitation",
        "AQI_PM10"
    ],

    "Orange": [
        "Avg Soil Temp (F)",
        "rodent_lbs_prd_used",
        "FIRE_Acres_Burned",
        "windeventcount"
    ],

    "San Diego": [
        "windeventcount",
        "fungicide_lbs_prd_used",
        "Avg Soil Temp (F)",
        "AQI_PM10"
    ],

    "San Luis Obispo": [
        "fungicide_lbs_prd_used",
        "Avg Soil Temp (F)",
        "Avg Rel Hum (%)",
        "windeventcount"
    ],

    "Tulare": [
        "Avg Soil Temp (F)",
        "Precipitation",
        "windeventcount",
        "Avg Wind Speed (mph)"
    ],

    "Ventura": [
        "AQI_PM10",
        "FIRE_Acres_Burned",
        "Avg Rel Hum (%)"
    ]
}

In [62]:
aggregated_data = pd.read_csv('/content/drive/MyDrive/DSSI/Project/Valley Fever - Environmental/data_processed/aggregate/aggregate_all_counties_2001_2024.csv', parse_dates=['Year-Month'])
aggregated_data['log1p_rate_per_100k'] = np.log1p(aggregated_data['rate_per_100k'])
aggregated_data['log1p_diff_rate_per_100k'] = aggregated_data['log1p_rate_per_100k'].diff()

In [63]:
def evaluate_lagged_feature(
    county_df,
    feature,
    lag,
    target_col="rate_log1p_diff",
    date_col="Year-Month",
    n_splits=5
):
    model_df = county_df[
        [date_col, target_col, feature]
    ].copy()

    lagged_feature = f"{feature}_lag_{lag}"

    model_df[lagged_feature] = (
        model_df[feature].shift(lag)
    )

    model_df = (
        model_df[
            [date_col, target_col, lagged_feature]
        ]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .reset_index(drop=True)
    )

    X = model_df[[lagged_feature]]
    y = model_df[target_col]

    tscv = TimeSeriesSplit(n_splits=n_splits)

    fold_rmses = []
    fold_maes = []
    fold_r2s = []

    for train_index, validation_index in tscv.split(X):

        X_train = X.iloc[train_index]
        X_validation = X.iloc[validation_index]

        y_train = y.iloc[train_index]
        y_validation = y.iloc[validation_index]

        model = LinearRegression()
        model.fit(X_train, y_train)

        predictions = model.predict(X_validation)

        fold_rmses.append(
            np.sqrt(
                mean_squared_error(
                    y_validation,
                    predictions
                )
            )
        )

        fold_maes.append(
            mean_absolute_error(
                y_validation,
                predictions
            )
        )

        fold_r2s.append(
            r2_score(
                y_validation,
                predictions
            )
        )

    return {
        "RMSE_Mean": np.mean(fold_rmses),
        "RMSE_Std": np.std(fold_rmses),
        "MAE_Mean": np.mean(fold_maes),
        "R2_Mean": np.mean(fold_r2s),
        "Fold_RMSEs": fold_rmses
    }

In [64]:
lag_results = []

test_months = 12

for county, features in selected_features_by_county.items():

    county_df = (
        aggregated_data[aggregated_data["County"] == county]
        .copy()
        .sort_values("Year-Month")
        .reset_index(drop=True)
    )

    # Keep final 12 months untouched
    development_df = (
        county_df
        .iloc[:-test_months]
        .copy()
        .reset_index(drop=True)
    )

    for feature in features:

        for lag in range(1, 13):

            results = evaluate_lagged_feature(
                county_df=development_df,
                feature=feature,
                lag=lag,
                target_col='log1p_diff_rate_per_100k',
                date_col="Year-Month",
                n_splits=5
            )

            lag_results.append({
                "County": county,
                "Feature": feature,
                "Lag": lag,
                "RMSE_Mean": results["RMSE_Mean"],
                "RMSE_Std": results["RMSE_Std"],
                "MAE_Mean": results["MAE_Mean"],
                "R2_Mean": results["R2_Mean"],
                "Fold_RMSEs": results["Fold_RMSEs"]
            })

In [65]:
feature_lag_results = pd.DataFrame(lag_results)

In [66]:
feature_lag_results = (
    feature_lag_results
    .sort_values(
        ["County", "Feature", "RMSE_Mean", "RMSE_Std"]
    )
    .reset_index(drop=True)
)

feature_lag_results["Lag_Rank"] = (
    feature_lag_results
    .groupby(["County", "Feature"])
    .cumcount() + 1
)

In [67]:
best_lag_per_feature = (
    feature_lag_results[
        feature_lag_results["Lag_Rank"] == 1
    ]
    .reset_index(drop=True)
)

display(best_lag_per_feature)

,County,Feature,Lag,RMSE_Mean,RMSE_Std,MAE_Mean,R2_Mean,Fold_RMSEs,Lag_Rank
0,Fresno,AQI_PM10,5,0.265003,0.060086,0.212836,0.029047,"[0.36298208907429863, 0.2759068516087297, 0.27...",1
1,Fresno,Avg Soil Temp (F),1,0.261233,0.061779,0.208395,0.060546,"[0.3615725933038338, 0.26873640640675744, 0.27...",1
2,Fresno,Wind Run (miles),3,0.262846,0.061181,0.211534,0.048215,"[0.36087137840768146, 0.2706360013552615, 0.27...",1
3,Fresno,fungicide_lbs_prd_used,10,0.262588,0.063227,0.213510,0.037248,"[0.3692897344877586, 0.2636651773533673, 0.271...",1
4,Fresno,windeventcount,9,0.269200,0.066352,0.219920,0.002088,"[0.37434405094004203, 0.27395911467534073, 0.2...",1
5,Kern,Avg Rel Hum (%),7,0.240391,0.043821,0.194073,0.057143,"[0.3077892295293016, 0.27305715491881877, 0.18...",1
6,Kern,Avg Soil Temp (F),1,0.238585,0.054525,0.193280,0.110587,"[0.31210482917837795, 0.2882971897481087, 0.18...",1
7,Kern,fungicide_lbs_prd_used,4,0.241918,0.058654,0.194101,0.088946,"[0.33218476812121445, 0.2801500857758102, 0.19...",1
8,Los Angeles,AQI_PM10,5,0.071766,0.019492,0.055837,0.006233,"[0.046862679385074844, 0.049004716790196245, 0...",1
9,Los Angeles,Avg Wind Speed (mph),1,0.071836,0.019476,0.055832,0.003962,"[0.047290906840515, 0.04875491475711058, 0.088...",1


In [68]:
top_3_lags_per_feature = (
    feature_lag_results[
        feature_lag_results["Lag_Rank"] <= 3
    ]
    .reset_index(drop=True)
)

display(top_3_lags_per_feature)

,County,Feature,Lag,RMSE_Mean,RMSE_Std,MAE_Mean,R2_Mean,Fold_RMSEs,Lag_Rank
0,Fresno,AQI_PM10,5,0.265003,0.060086,0.212836,0.029047,"[0.36298208907429863, 0.2759068516087297, 0.27...",1
1,Fresno,AQI_PM10,3,0.266202,0.059572,0.216041,0.015009,"[0.368012927743354, 0.28061770454779333, 0.258...",2
2,Fresno,AQI_PM10,6,0.267727,0.060582,0.215627,0.010382,"[0.3647124549149305, 0.2732488892153794, 0.277...",3
3,Fresno,Avg Soil Temp (F),1,0.261233,0.061779,0.208395,0.060546,"[0.3615725933038338, 0.26873640640675744, 0.27...",1
4,Fresno,Avg Soil Temp (F),12,0.263393,0.067038,0.214528,0.045514,"[0.37324911406546485, 0.26598784552904287, 0.2...",2
...,...,...,...,...,...,...,...,...,...
85,Ventura,Avg Rel Hum (%),3,0.275928,0.024953,0.221175,0.010785,"[0.28727509504103815, 0.2584875685302001, 0.26...",2
86,Ventura,Avg Rel Hum (%),5,0.276627,0.026698,0.218221,0.006715,"[0.2885520652366771, 0.2539287458384571, 0.268...",3
87,Ventura,FIRE_Acres_Burned,5,0.275650,0.032408,0.215825,-0.002759,"[0.25976047860625767, 0.25934738300296833, 0.3...",1
88,Ventura,FIRE_Acres_Burned,1,0.275653,0.032280,0.216537,-0.002877,"[0.25989045693693413, 0.25962300024482293, 0.3...",2


## Searching joint lag combinations now

In [69]:
import itertools

In [70]:
candidate_lags_by_county = {
    "Fresno": {
        "windeventcount": [9, 3, 1],
        "fungicide_lbs_prd_used": [10, 1, 7],
        "AQI_PM10": [5, 3, 6],
        "Avg Soil Temp (F)": [1, 12, 7],
        "Wind Run (miles)": [3, 2, 8]
    },

    "Kern": {
        "Avg Rel Hum (%)": [7, 1, 12],
        "fungicide_lbs_prd_used": [4, 3, 9],
        "Avg Soil Temp (F)": [1, 7, 12]
    },

    "Los Angeles": {
        "Avg Wind Speed (mph)": [1, 9, 2],
        "Precipitation": [5, 6, 1],
        "AQI_PM10": [5, 6, 1]
    },

    "Orange": {
        "Avg Soil Temp (F)": [7, 8, 12],
        "rodent_lbs_prd_used": [7, 4, 3],
        "FIRE_Acres_Burned": [6, 3, 11],
        "windeventcount": [9, 10, 12]
    },

    "San Diego": {
        "windeventcount": [3, 8, 4],
        "fungicide_lbs_prd_used": [1, 3, 5],
        "Avg Soil Temp (F)": [5, 4, 3],
        "AQI_PM10": [2, 5, 1]
    },

    "San Luis Obispo": {
        "fungicide_lbs_prd_used": [2, 8, 3],
        "Avg Soil Temp (F)": [1, 7, 2],
        "Avg Rel Hum (%)": [4, 3, 6],
        "windeventcount": [3, 8, 1]
    },

    "Tulare": {
        "Avg Soil Temp (F)": [6, 12, 8],
        "Precipitation": [12, 6, 11],
        "windeventcount": [12, 11, 6],
        "Avg Wind Speed (mph)": [2, 12, 11]
    },

    "Ventura": {
        "AQI_PM10": [1, 2, 3],
        "FIRE_Acres_Burned": [5, 1, 3],
        "Avg Rel Hum (%)": [2, 3, 5]
    }
}

In [71]:
def evaluate_joint_lag_model(
    county_df,
    feature_lag_map,
    target_col="rate_log1p_diff",
    date_col="Year-Month",
    n_splits=5
):
    """
    Evaluate one specific feature-lag assignment using time-series CV.

    Example feature_lag_map:
    {
        "AQI_PM10": 3,
        "Precipitation": 5,
        "Avg Soil Temp (F)": 2
    }
    """

    required_columns = (
        [date_col, target_col]
        + list(feature_lag_map.keys())
    )

    model_df = county_df[required_columns].copy()

    lagged_columns = []

    for feature, lag in feature_lag_map.items():

        lagged_column = f"{feature}_lag_{lag}"

        model_df[lagged_column] = (
            model_df[feature].shift(lag)
        )

        lagged_columns.append(lagged_column)

    model_df = (
        model_df[
            [date_col, target_col]
            + lagged_columns
        ]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .reset_index(drop=True)
    )

    X = model_df[lagged_columns]
    y = model_df[target_col]

    tscv = TimeSeriesSplit(n_splits=n_splits)

    fold_rmses = []
    fold_maes = []
    fold_r2s = []

    for train_index, validation_index in tscv.split(X):

        X_train = X.iloc[train_index]
        X_validation = X.iloc[validation_index]

        y_train = y.iloc[train_index]
        y_validation = y.iloc[validation_index]

        model = LinearRegression()

        model.fit(X_train, y_train)

        predictions = model.predict(X_validation)

        fold_rmses.append(
            np.sqrt(
                mean_squared_error(
                    y_validation,
                    predictions
                )
            )
        )

        fold_maes.append(
            mean_absolute_error(
                y_validation,
                predictions
            )
        )

        fold_r2s.append(
            r2_score(
                y_validation,
                predictions
            )
        )

    return {
        "RMSE_Mean": np.mean(fold_rmses),
        "RMSE_Std": np.std(fold_rmses),
        "MAE_Mean": np.mean(fold_maes),
        "MAE_Std": np.std(fold_maes),
        "R2_Mean": np.mean(fold_r2s),
        "R2_Std": np.std(fold_r2s),
        "Fold_RMSEs": fold_rmses,
        "Fold_MAEs": fold_maes,
        "Fold_R2s": fold_r2s,
        "Number_Observations": len(model_df)
    }

In [76]:
joint_lag_results = []

test_months = 12

for county, feature_lag_options in candidate_lags_by_county.items():

    print(f"Processing {county}...")

    county_df = (
        aggregated_data[
            aggregated_data["County"] == county
        ]
        .copy()
        .sort_values("Year-Month")
        .reset_index(drop=True)
    )

    # Keep final 12 months untouched
    development_df = (
        county_df
        .iloc[:-test_months]
        .copy()
        .reset_index(drop=True)
    )

    features = list(feature_lag_options.keys())

    lag_option_lists = [
        feature_lag_options[feature]
        for feature in features
    ]

    # Cartesian product of candidate lag choices
    lag_combinations = itertools.product(
        *lag_option_lists
    )

    for lag_values in lag_combinations:

        feature_lag_map = dict(
            zip(features, lag_values)
        )

        try:
            results = evaluate_joint_lag_model(
                county_df=development_df,
                feature_lag_map=feature_lag_map,
                target_col="log1p_diff_rate_per_100k",
                date_col="Year-Month",
                n_splits=5
            )

            joint_lag_results.append({
                "County": county,

                "Feature_Lag_Map":
                    feature_lag_map.copy(),

                "Feature_Lag_Text":
                    " + ".join(
                        f"{feature} lag {lag}"
                        for feature, lag
                        in feature_lag_map.items()
                    ),

                "Number_Features":
                    len(feature_lag_map),

                "RMSE_Mean":
                    results["RMSE_Mean"],

                "RMSE_Std":
                    results["RMSE_Std"],

                "MAE_Mean":
                    results["MAE_Mean"],

                "MAE_Std":
                    results["MAE_Std"],

                "R2_Mean":
                    results["R2_Mean"],

                "R2_Std":
                    results["R2_Std"],

                "Number_Observations":
                    results["Number_Observations"],

                "Fold_RMSEs":
                    results["Fold_RMSEs"],

                "Fold_MAEs":
                    results["Fold_MAEs"],

                "Fold_R2s":
                    results["Fold_R2s"]
            })

        except Exception as error:
            print(
                county,
                feature_lag_map,
                error
            )

Processing Fresno...
Processing Kern...
Processing Los Angeles...
Processing Orange...
Processing San Diego...
Processing San Luis Obispo...
Processing Tulare...
Processing Ventura...


In [77]:
joint_lag_results_df = pd.DataFrame(joint_lag_results)
joint_lag_results_df

,County,Feature_Lag_Map,Feature_Lag_Text,Number_Features,RMSE_Mean,RMSE_Std,MAE_Mean,MAE_Std,R2_Mean,R2_Std,Number_Observations,Fold_RMSEs,Fold_MAEs,Fold_R2s
0,Fresno,"{'windeventcount': 9, 'fungicide_lbs_prd_used'...",windeventcount lag 9 + fungicide_lbs_prd_used ...,5,0.267806,0.067778,0.218752,0.059558,-0.003090,0.150952,266,"[0.3857383392031942, 0.2653165334097149, 0.278...","[0.32387869608725245, 0.20418677356912057, 0.2...","[-0.05808134019069433, 0.047430940674466404, -..."
1,Fresno,"{'windeventcount': 9, 'fungicide_lbs_prd_used'...",windeventcount lag 9 + fungicide_lbs_prd_used ...,5,0.276051,0.087091,0.222645,0.072747,-0.041515,0.196561,266,"[0.4361635719771087, 0.26653867731723635, 0.27...","[0.3558934446808901, 0.21393793890970414, 0.22...","[-0.35279573238416706, 0.038634973334725875, 0..."
2,Fresno,"{'windeventcount': 9, 'fungicide_lbs_prd_used'...",windeventcount lag 9 + fungicide_lbs_prd_used ...,5,0.274202,0.083003,0.222566,0.069861,-0.030971,0.176254,266,"[0.4252245177478205, 0.2661088811866762, 0.274...","[0.34933994480533226, 0.21362971903525718, 0.2...","[-0.28578998113823273, 0.041732893948015826, 0..."
3,Fresno,"{'windeventcount': 9, 'fungicide_lbs_prd_used'...",windeventcount lag 9 + fungicide_lbs_prd_used ...,5,0.275653,0.084724,0.222895,0.071630,-0.039544,0.181884,264,"[0.4287820097539054, 0.27214109334402076, 0.27...","[0.3527301066340415, 0.21250570751084732, 0.23...","[-0.307394199883378, -0.0022039105823896676, 0..."
4,Fresno,"{'windeventcount': 9, 'fungicide_lbs_prd_used'...",windeventcount lag 9 + fungicide_lbs_prd_used ...,5,0.277189,0.088234,0.223609,0.074734,-0.053108,0.214729,264,"[0.4408302799886643, 0.2709606767947534, 0.267...","[0.3622978785715821, 0.21610486184286823, 0.21...","[-0.3818989241065398, 0.006471386388878231, 0...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
643,Ventura,"{'AQI_PM10': 3, 'FIRE_Acres_Burned': 1, 'Avg R...",AQI_PM10 lag 3 + FIRE_Acres_Burned lag 1 + Avg...,3,0.272965,0.032037,0.219062,0.028444,0.015875,0.041993,208,"[0.2572319612512965, 0.2634893489130492, 0.335...","[0.20999612903485915, 0.20766726922116707, 0.2...","[0.01937331363224859, -0.03536684896996034, 0...."
644,Ventura,"{'AQI_PM10': 3, 'FIRE_Acres_Burned': 1, 'Avg R...",AQI_PM10 lag 3 + FIRE_Acres_Burned lag 1 + Avg...,3,0.276075,0.031079,0.221824,0.023847,-0.010433,0.092052,208,"[0.2482784065944007, 0.2819446302261687, 0.334...","[0.20079334417079764, 0.22404690123069837, 0.2...","[0.08645120566200903, -0.18548423244100487, 0...."
645,Ventura,"{'AQI_PM10': 3, 'FIRE_Acres_Burned': 3, 'Avg R...",AQI_PM10 lag 3 + FIRE_Acres_Burned lag 3 + Avg...,3,0.281642,0.036406,0.223322,0.036565,-0.068912,0.259200,206,"[0.32629918075970854, 0.2579479761586259, 0.32...","[0.27350477621970576, 0.19738122074180375, 0.2...","[-0.5779230365032817, 0.007724253649037727, 0...."
646,Ventura,"{'AQI_PM10': 3, 'FIRE_Acres_Burned': 3, 'Avg R...",AQI_PM10 lag 3 + FIRE_Acres_Burned lag 3 + Avg...,3,0.273062,0.031989,0.219415,0.028447,0.015081,0.043283,206,"[0.2575599807072184, 0.2638323372153942, 0.335...","[0.21171383540371727, 0.2076978038719822, 0.27...","[0.01687074962570201, -0.038064110556870645, 0..."


In [78]:
joint_lag_results_df = (
    joint_lag_results_df
    .sort_values(
        [
            "County",
            "RMSE_Mean",
            "RMSE_Std",
            "MAE_Mean"
        ],
        ascending=[
            True,
            True,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)

joint_lag_results_df["Joint_Lag_Rank"] = (
    joint_lag_results_df
    .groupby("County")
    .cumcount() + 1
)

In [79]:
joint_lag_results_df = (
    joint_lag_results_df
    .sort_values(
        [
            "County",
            "RMSE_Mean",
            "RMSE_Std",
            "MAE_Mean"
        ],
        ascending=[
            True,
            True,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)

joint_lag_results_df["Joint_Lag_Rank"] = (
    joint_lag_results_df
    .groupby("County")
    .cumcount() + 1
)

In [86]:
top5_joint_lags = (
    joint_lag_results_df[
        joint_lag_results_df["Joint_Lag_Rank"] <=5
    ][
        [
            "County",
            "Feature_Lag_Map",
            "Feature_Lag_Text",
            "RMSE_Mean",
            "RMSE_Std",
            "MAE_Mean",
            "R2_Mean"
        ]
    ]
    .reset_index(drop=True)
)

display(top5_joint_lags)

,County,Feature_Lag_Map,Feature_Lag_Text,RMSE_Mean,RMSE_Std,MAE_Mean,R2_Mean
0,Fresno,"{'windeventcount': 3, 'fungicide_lbs_prd_used'...",windeventcount lag 3 + fungicide_lbs_prd_used ...,0.260981,0.063709,0.209993,0.061795
1,Fresno,"{'windeventcount': 1, 'fungicide_lbs_prd_used'...",windeventcount lag 1 + fungicide_lbs_prd_used ...,0.261869,0.071433,0.212392,0.065799
2,Fresno,"{'windeventcount': 3, 'fungicide_lbs_prd_used'...",windeventcount lag 3 + fungicide_lbs_prd_used ...,0.261918,0.072133,0.211949,0.065662
3,Fresno,"{'windeventcount': 3, 'fungicide_lbs_prd_used'...",windeventcount lag 3 + fungicide_lbs_prd_used ...,0.262030,0.069040,0.214199,0.054102
4,Fresno,"{'windeventcount': 3, 'fungicide_lbs_prd_used'...",windeventcount lag 3 + fungicide_lbs_prd_used ...,0.262146,0.069366,0.213843,0.056479
5,Kern,"{'Avg Rel Hum (%)': 7, 'fungicide_lbs_prd_used...",Avg Rel Hum (%) lag 7 + fungicide_lbs_prd_used...,0.235534,0.047582,0.191242,0.110376
6,Kern,"{'Avg Rel Hum (%)': 7, 'fungicide_lbs_prd_used...",Avg Rel Hum (%) lag 7 + fungicide_lbs_prd_used...,0.236039,0.042252,0.190592,0.090542
7,Kern,"{'Avg Rel Hum (%)': 1, 'fungicide_lbs_prd_used...",Avg Rel Hum (%) lag 1 + fungicide_lbs_prd_used...,0.238395,0.050399,0.191531,0.101665
8,Kern,"{'Avg Rel Hum (%)': 7, 'fungicide_lbs_prd_used...",Avg Rel Hum (%) lag 7 + fungicide_lbs_prd_used...,0.238933,0.046846,0.193138,0.081098
9,Kern,"{'Avg Rel Hum (%)': 1, 'fungicide_lbs_prd_used...",Avg Rel Hum (%) lag 1 + fungicide_lbs_prd_used...,0.239136,0.051103,0.191787,0.096615


In [ ]:
joint_lag_results_df["Best_County_RMSE"] = (
    joint_lag_results_df
    .groupby("County")["RMSE_Mean"]
    .transform("min")
)

competitive_joint_lags = joint_lag_results_df[
    joint_lag_results_df["RMSE_Mean"]
    <= joint_lag_results_df["Best_County_RMSE"] * 1.01
].copy()

In [84]:
selected_joint_lags = (
    competitive_joint_lags
    .sort_values(
        [
            "County",
            "RMSE_Std",
            "RMSE_Mean",
            "MAE_Mean"
        ]
    )
    .groupby("County", group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

display(
    selected_joint_lags[
        [
            "County",
            "Feature_Lag_Map",
            "Feature_Lag_Text",
            "RMSE_Mean",
            "RMSE_Std",
            "MAE_Mean",
            "R2_Mean"
        ]
    ]
)

,County,Feature_Lag_Map,Feature_Lag_Text,RMSE_Mean,RMSE_Std,MAE_Mean,R2_Mean
0,Fresno,"{'windeventcount': 3, 'fungicide_lbs_prd_used'...",windeventcount lag 3 + fungicide_lbs_prd_used ...,0.262800,0.063335,0.211971,0.046937
1,Kern,"{'Avg Rel Hum (%)': 7, 'fungicide_lbs_prd_used...",Avg Rel Hum (%) lag 7 + fungicide_lbs_prd_used...,0.236039,0.042252,0.190592,0.090542
2,Los Angeles,"{'Avg Wind Speed (mph)': 2, 'Precipitation': 6...",Avg Wind Speed (mph) lag 2 + Precipitation lag...,0.072097,0.018973,0.056247,-0.009062
3,Orange,"{'Avg Soil Temp (F)': 12, 'rodent_lbs_prd_used...",Avg Soil Temp (F) lag 12 + rodent_lbs_prd_used...,0.115215,0.007803,0.095261,-0.019047
4,San Diego,"{'windeventcount': 3, 'fungicide_lbs_prd_used'...",windeventcount lag 3 + fungicide_lbs_prd_used ...,0.116456,0.015831,0.095645,0.019236
5,San Luis Obispo,"{'fungicide_lbs_prd_used': 2, 'Avg Soil Temp (...",fungicide_lbs_prd_used lag 2 + Avg Soil Temp (...,0.443149,0.056859,0.355267,0.076137
6,Tulare,"{'Avg Soil Temp (F)': 8, 'Precipitation': 12, ...",Avg Soil Temp (F) lag 8 + Precipitation lag 12...,0.317812,0.045260,0.256834,0.046856
7,Ventura,"{'AQI_PM10': 1, 'FIRE_Acres_Burned': 1, 'Avg R...",AQI_PM10 lag 1 + FIRE_Acres_Burned lag 1 + Avg...,0.268059,0.027757,0.215016,0.044817


In [87]:
# saving
top5_joint_lags.to_csv('top5_joint_lags.csv', index=False)
selected_joint_lags.to_csv('best_joint_lags.csv', index=False)